In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch 
import numpy as np
import random
torch.autograd.set_detect_anomaly(True)
torch.multiprocessing.set_sharing_strategy("file_descriptor")
seed = 140421
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

In [ ]:
import os
KITTI_ROOT = os.path.join("data", "Kitty")
IMAGE_DIR = os.path.join(KITTI_ROOT, "data_object_image_2", "training", "image_2")
LABEL_DIR = os.path.join(KITTI_ROOT, "data_object_label_2", "training", "label_2")
OUTPUT_JSON = os.path.join("data", "Kitty", "kitti_coco.json")

In [ ]:
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.data.datasets.builtin_meta import _get_builtin_metadata
from detectron2.data.datasets.coco_scale import KITTICocoDataset

kitty_dataset_name = "Kitty_train"
debug = True
debug_size = 100
DatasetCatalog.register(
    kitty_dataset_name,
    KITTICocoDataset(
        OUTPUT_JSON,
        IMAGE_DIR,
        debug=debug,
        debug_size=debug_size
    )
)
coco_meta = _get_builtin_metadata("coco")
MetadataCatalog.get(kitty_dataset_name).set(
    thing_dataset_id_to_contiguous_id = {1: 0, 3:2}  # COCO ID 1 → internal ID 0
)

# Register Calib

In [ ]:
from detectron2.data.datasets.pano360 import CalibDataset

debug = True
calib_train = CalibDataset(
    train=True,
    json_name="datasets/pano360_crops_dataset_cvpr_myDistWider_train.json",
    logger=None,
    debug=debug,
)
calib_val = CalibDataset(
    train=False,
    json_name="datasets/pano360_crops_dataset_cvpr_myDistWider_train.json",
    logger=None,
    debug=debug,
)

In [ ]:
from detectron2.data import DatasetCatalog
pano_train_name = "Pano360_train"
pano_val_name = "Pano360_val"
DatasetCatalog.register(pano_train_name, calib_train)
DatasetCatalog.register(pano_val_name, calib_val)

In [ ]:
import os
from detectron2.engine import KittyCalibTrainer
from detectron2 import model_zoo
from detectron2.config import get_cfg

cfg = get_cfg()
# config_path = "COCO-Keypoints/keypoint_rcnn_R_50_FPN_3x.yaml"
config_path = "COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml"
cfg.merge_from_file(model_zoo.get_config_file(config_path))
experiment_name = "coco-scale-roih-biasl2"
# cfg.MODEL.CAMERA_HEAD.NUM_CONV = 0
# cfg.MODEL.CAMERA_HEAD.NUM_FC = 1
cfg.OUTPUT_DIR = os.path.join("output", experiment_name)
#cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(config_path)  # Let training initialize from model zoo
cfg.MODEL.WEIGHTS = os.path.join("output", experiment_name, "model_final.pth")
cfg.SOLVER.IMS_PER_BATCH = 4 # This is the real "batch size" commonly known to deep learning people
cfg.SOLVER.BASE_LR = 0.00025  # pick a good LR
cfg.MODEL.KEYPOINT_ON = False
cfg.MODEL.POINT_NET_ON = False # In kitty we know the camera_height is ~1.65
cfg.MODEL.HEIGHT_ON = True 
cfg.MODEL.HEIGHT_REFINE_ON = True
cfg.MODEL.META_ARCHITECTURE = "GeneralizedCamRCNN"
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1  # Number of classes
cfg.MODEL.ROI_KEYPOINT_HEAD.NUM_KEYPOINTS = 17 # Number of keypoints
cfg.MODEL.ROI_HEADS.NAME = "HeightStandardROIHeads"
cfg.MODEL.ROI_KEYPOINT_HEAD.NAME = "KRCNNConvDeconvUpsampleHead"
cfg.VIS_PERIOD = 10
cfg.DATALOADER.FILTER_EMPTY_ANNOTATIONS = False  # Dataset is filtered before entering MixedDataset with build_detection_dataset.
cfg.DATALOADER.ASPECT_RATIO_GROUPING = False  # Doesn't work at the current implementation of MixedDataset
cfg.SOLVER.MAX_ITER = 5000
# cfg.SOLVER.WARMUP_ITERS = 0
cfg.SOLVER.AMP.ENABLED = True  # Enable AMP here -- improve 10s per iter approx.
cfg.SOLVER.RATIO_PANO360 = (3, 1)
cfg.FLOAT32_PRECISION = "medium"
# cfg.DATALOADER
cfg.DATASETS.TRAIN = (kitty_dataset_name, )
cfg.DATASETS.TEST = (kitty_dataset_name, )
cfg.DATALOADER.NUM_WORKERS = 4

# SVMIW Losses
cfg.MODEL.ROI_KEYPOINT_HEAD.LOSS_WEIGHT = 10  # alpha_4
cfg.MODEL.ROI_BOX_HEAD.BBOX_REG_LOSS_WEIGHT = 10  # alpha_5
cfg.MODEL.HEIGHT_HEAD.LOSS_WEIGHT = 0.05  # alpha 2
cfg.MODEL.HEIGHT_HEAD.REDUCE_METHOD = "softmax"
cfg.MODEL.HEIGHT_HEAD.SMOOTH_L1_BETA = 0.1

cfg.MODEL.CAMERA_HEAD.LOSS_CRITERION = "softargmax_l2_biased"

cfg.MODEL.POINT_NET.POOLING="max"
cfg.MODEL.POINT_NET.TEMPERATURE = 1.0
# NOTE: Check these values
cfg.MODEL.POINT_NET.BN = False
cfg.MODEL.POINT_NET.TRANSFORM = True
cfg.MODEL.HEIGHT_REFINE_ON = True
cfg.MODEL.POINT_NET.DETACH = False
cfg.MODEL.POINT_NET.REFINE_TEMPERATURE = 1.0


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "garbage_collection_threshold:0.6,max_split_size_mb:128,expandable_segments:True"
trainer = KittyCalibTrainer(cfg) 
trainer.resume_or_load(resume=False)
model = trainer.model
# trainer.train()

In [ ]:
trainer.test(cfg, model)